<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/05-files-and-data/03_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 5: Solutions

Try the exercises first. The notes after each solution explain the decisions that matter with real data files: encodings, missing values, ordering, and not losing or changing data by accident.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import copy
import math
import csv
import inspect
import json
import tempfile
from pathlib import Path


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, float) or isinstance(b, float):
        if not isinstance(a, (int, float)) or not isinstance(b, (int, float)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=70):
    text = repr(x)
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


def _tmp() -> Path:
    """A fresh, empty temporary folder for one test."""
    return Path(tempfile.mkdtemp())


def _write(path, text, encoding="utf-8"):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding=encoding)
    return path


# ---- checks ----

def check_parse_value(f):
    typed = lambda text: (f(text), type(f(text)).__name__)
    _scenarios("parse_value", [
        ("integer", lambda: typed("42"), (42, "int"), "try int() before float()"),
        ("float", lambda: typed("3.5"), (3.5, "float"), None),
        ("scientific notation", lambda: f("1e-3"), 0.001, None),
        ("negative", lambda: f("-7"), -7, None),
        ("NA with spaces", lambda: f("  NA "), None, "strip first, then compare in lowercase"),
        ("empty string", lambda: f(""), None, None),
        ("question mark", lambda: f("?"), None, None),
        ("the text 'nan'", lambda: f("nan"), None, "float('nan') works, so check for missing markers before converting"),
        ("TRUE", lambda: typed("TRUE"), (True, "bool"), None),
        ("false", lambda: typed("false"), (False, "bool"), None),
        ("text is stripped", lambda: f("  Adelie "), "Adelie", None),
        ("text that starts like a number", lambda: f("3 g"), "3 g", None),
    ])


def check_count_lines(f):
    def count(text):
        return f(_write(_tmp() / "f.txt", text))

    _scenarios("count_lines", [
        ("three lines with a final newline", lambda: count("a\nb\nc\n"), 3, None),
        ("last line without a newline", lambda: count("a\nb\nc"), 3, "don't count newline characters; count lines"),
        ("empty file", lambda: count(""), 0, None),
        ("blank lines count too", lambda: count("a\n\nb\n"), 3, None),
        ("accepts a string path", lambda: f(str(_write(_tmp() / "g.txt", "x\ny\n"))), 2, None),
    ])


def check_save_json(f):
    def content():
        path = _tmp() / "out" / "nested" / "result.json"
        f({"b": 1, "a": [1, 2]}, path)
        return path.read_text(encoding="utf-8")

    def returns_path():
        path = _tmp() / "x.json"
        result = f({}, path)
        return isinstance(result, Path) and result == path

    def unicode():
        path = _tmp() / "u.json"
        f({"species": "Adélie"}, path)
        return "Adélie" in path.read_text(encoding="utf-8")

    def round_trip():
        path = _tmp() / "r.json"
        data = {"model": "knn", "params": {"k": 5}, "scale": True, "notes": None}
        f(data, path)
        return json.loads(path.read_text(encoding="utf-8")) == data

    _scenarios("save_json", [
        ("creates missing folders", lambda: (content() is not None), True, "path.parent.mkdir(parents=True, exist_ok=True)"),
        ("exact formatting", content, '{\n  "b": 1,\n  "a": [\n    1,\n    2\n  ]\n}\n',
         "indent=2, keep the key order, and end the file with a newline"),
        ("returns the path as a Path", returns_path, True, None),
        ("non-English text is written as-is", unicode, True, "ensure_ascii=False"),
        ("round trip", round_trip, True, None),
    ])


def check_find_files(f):
    def setup():
        root = _tmp()
        for name in ["a.csv", "b.txt", "sub/c.csv", "sub/deep/d.CSV", "sub/e.json"]:
            _write(root / name, "x")
        (root / "folder.csv").mkdir()
        return root

    _scenarios("find_files", [
        ("finds files in all subfolders", lambda: f(setup(), ".csv"), ["a.csv", "sub/c.csv", "sub/deep/d.CSV"],
         "search recursively (rglob), compare suffixes in lowercase, skip folders, return sorted relative paths"),
        ("other suffix", lambda: f(setup(), ".json"), ["sub/e.json"], None),
        ("suffix given in capitals", lambda: f(setup(), ".TXT"), ["b.txt"], None),
        ("nothing found", lambda: f(setup(), ".png"), [], None),
        ("accepts a string path", lambda: f(str(setup()), ".txt"), ["b.txt"], None),
    ])


def check_read_csv_records(f):
    def basic():
        path = _write(_tmp() / "people.csv", 'name,age,city\n"Smith, John",42,Kathmandu\nAna,NA,Pokhara\n')
        return f(path)

    def bom():
        path = _write(_tmp() / "excel.csv", "id,score\n1,0.5\n", encoding="utf-8-sig")
        return f(path)

    def empty_file():
        return f(_write(_tmp() / "e.csv", "id,score\n"))

    _scenarios("read_csv_records", [
        ("quoted commas and missing values", basic,
         [{"name": "Smith, John", "age": 42, "city": "Kathmandu"}, {"name": "Ana", "age": None, "city": "Pokhara"}],
         "use csv.DictReader, and convert every value with parse_value"),
        ("a file saved by Excel (with a BOM)", bom, [{"id": 1, "score": 0.5}],
         "open with encoding='utf-8-sig' so the first column isn't called '\\ufeffid'"),
        ("header only", empty_file, [], None),
    ])


def check_write_csv(f):
    def read_back(path):
        with open(path, encoding="utf-8", newline="") as fh:
            return list(csv.reader(fh))

    def basic():
        path = _tmp() / "out.csv"
        f([{"name": "Smith, John", "age": 42}, {"name": "Ana", "city": "Pokhara"}], path)
        return read_back(path)

    def returns_count():
        return f([{"a": 1}, {"a": 2}, {"a": 3}], _tmp() / "n.csv")

    def nones():
        path = _tmp() / "none.csv"
        f([{"a": None, "b": 0}], path)
        return read_back(path)

    _scenarios("write_csv", [
        ("columns from all records, in first-seen order", basic,
         [["name", "age", "city"], ["Smith, John", "42", ""], ["Ana", "", "Pokhara"]],
         "collect the keys from every record. Missing keys and None become empty fields"),
        ("None becomes empty, 0 stays 0", nones, [["a", "b"], ["", "0"]], "check `is None`, not truthiness"),
        ("returns the number of rows written", returns_count, 3, None),
    ])


def check_missing_report(f):
    records = [
        {"species": "Adelie", "mass": 3700, "sex": None},
        {"species": "Gentoo", "mass": None, "sex": None},
        {"species": "Gentoo", "mass": 5000},
        {"species": None, "mass": 3500, "sex": "male", "notes": "tagged"},
    ]
    _scenarios("missing_report", [
        ("counts, fractions and order", lambda: f(records),
         [("notes", 3, 0.75), ("sex", 3, 0.75), ("mass", 1, 0.25), ("species", 1, 0.25)],
         "a key that's absent from a record counts as missing too. Sort by count (high to low), then by name"),
        ("nothing missing", lambda: f([{"a": 1}, {"a": 2}]), [("a", 0, 0.0)], None),
        ("no records", lambda: f([]), [], None),
    ])


def check_fill_missing(f):
    train = [{"x": 1.0}, {"x": None}, {"x": 3.0}, {"x": 10.0}]
    test = [{"x": None}, {"x": 5.0}]

    def no_mutation():
        tr = [dict(r) for r in train]
        te = [dict(r) for r in test]
        f(tr, te, "x", "median")
        return (tr, te) == (train, test)

    def mode_text():
        tr = [{"c": "a"}, {"c": "b"}, {"c": "b"}, {"c": None}]
        return f(tr, [{"c": None}], "c", "mode")

    _scenarios("fill_missing", [
        ("median from train only", lambda: f(train, test, "x", "median"),
         ([{"x": 1.0}, {"x": 3.0}, {"x": 3.0}, {"x": 10.0}], [{"x": 3.0}, {"x": 5.0}], 3.0),
         "use only the non-missing training values: [1, 3, 10] has median 3"),
        ("mean", lambda: f(train, test, "x", "mean")[2], 14 / 3, None),
        ("even number of values", lambda: f([{"x": 1}, {"x": 2}, {"x": 4}, {"x": 9}], [], "x", "median")[2], 3.0,
         "with an even count, the median is the mean of the two middle values"),
        ("mode works for text", mode_text, ([{"c": "a"}, {"c": "b"}, {"c": "b"}, {"c": "b"}], [{"c": "b"}], "b"), None),
        ("inputs are not modified", no_mutation, True, "copy each record before changing it"),
        ("unknown strategy", lambda: f(train, test, "x", "average"), Raises(ValueError), None),
        ("column with no values in train", lambda: f([{"x": None}], [], "x", "mean"), Raises(ValueError), None),
    ])


def check_read_jsonl(f):
    def basic():
        path = _write(_tmp() / "log.jsonl", '{"epoch": 0, "loss": 0.9}\n\n{"epoch": 1, "loss": 0.5}\n')
        return list(f(path))

    def bad_line():
        path = _write(_tmp() / "bad.jsonl", '{"a": 1}\n{"a": 2}\n{"a": 3,,}\n')
        try:
            list(f(path))
        except ValueError as e:
            return "line 3" in str(e)
        return "no error raised"

    _scenarios("read_jsonl", [
        ("is a generator", lambda: inspect.isgenerator(f(_write(_tmp() / "g.jsonl", "{}\n"))), True, "use yield"),
        ("reads records and skips blank lines", basic, [{"epoch": 0, "loss": 0.9}, {"epoch": 1, "loss": 0.5}], None),
        ("bad JSON raises ValueError mentioning the line number", bad_line, True,
         "catch json.JSONDecodeError and raise ValueError(f'line {n}: ...')"),
    ])


def check_validate_records(f):
    schema = {
        "species": {"type": "str", "allowed": ["Adelie", "Chinstrap", "Gentoo"]},
        "body_mass_g": {"type": "float", "min": 1000, "max": 10000},
        "year": {"type": "int"},
        "sex": {"type": "str", "allowed": ["male", "female"], "required": False},
    }
    good = {"species": "Adelie", "body_mass_g": 3750.0, "year": 2007, "sex": "male"}
    records = [
        good,
        {"species": "Emperor", "body_mass_g": 3750, "year": 2007, "sex": None},
        {"species": "Gentoo", "body_mass_g": 50.0, "year": "2008"},
        {"species": "Gentoo", "body_mass_g": None, "year": 2008, "sex": "unknown"},
        {"species": "Gentoo", "body_mass_g": True, "year": 2009.0},
    ]
    _scenarios("validate_records", [
        ("a valid record passes", lambda: f([good], schema), ([good], []), None),
        ("every problem is reported", lambda: f(records, schema)[1], [
            (1, "species", "not_allowed"),
            (2, "body_mass_g", "below_min"),
            (2, "year", "type"),
            (3, "body_mass_g", "missing"),
            (3, "sex", "not_allowed"),
            (4, "body_mass_g", "type"),
            (4, "year", "type"),
        ], "check columns in schema order. An int is fine where a float is expected, but a bool never counts as a number"),
        ("only records with no errors are kept", lambda: f(records, schema)[0], [good], None),
        ("optional columns may be missing", lambda: f([{"species": "Gentoo", "body_mass_g": 5000, "year": 2008}], schema)[1], [],
         "sex has required=False, so None or an absent key is fine"),
    ])


def check_deduplicate(f):
    records = [
        {"id": "P1", "species": "Adelie", "mass": 3700},
        {"id": "P2", "species": "Gentoo", "mass": 5000},
        {"id": " p1 ", "species": "ADELIE", "mass": 3710},
        {"id": "P3", "species": "Adelie", "mass": 3700},
    ]

    def no_mutation():
        copies = [dict(r) for r in records]
        f(copies, ["id"])
        return copies == records

    _scenarios("deduplicate", [
        ("by id, ignoring case and spaces", lambda: f(records, ["id"]), ([records[0], records[1], records[3]], 1),
         "compare stripped, lowercased strings, but keep the original records unchanged in the output"),
        ("by several columns", lambda: f(records, ["species", "mass"]), ([records[0], records[1], records[2]], 1), None),
        ("all columns when keys is None", lambda: f(records, None), (records, 0),
         "with keys=None, compare every column. No two records match on all of them"),
        ("keeps the first occurrence", lambda: f([{"a": 1, "n": "x"}, {"a": 1, "n": "y"}], ["a"])[0], [{"a": 1, "n": "x"}], None),
        ("inputs are not modified", no_mutation, True, None),
    ])


def check_Checkpointer(C):
    def rotation():
        folder = _tmp() / "ckpt"
        c = C(folder, keep_last=3)
        for step in [1, 2, 3, 4, 5]:
            c.save(step, {"w": step * 0.1})
        return (c.steps(), sorted(p.name for p in folder.iterdir()))

    def latest():
        c = C(_tmp() / "ckpt", keep_last=2)
        c.save(10, {"w": 1.0})
        c.save(20, {"w": 2.0})
        return c.latest()

    def persists():
        folder = _tmp() / "ckpt"
        C(folder).save(7, {"lr": 0.01})
        return C(folder).latest()

    def ignores_other_files():
        folder = _tmp() / "ckpt"
        _write(folder / "notes.txt", "keep me")
        c = C(folder, keep_last=1)
        c.save(1, {})
        c.save(2, {})
        return sorted(p.name for p in folder.iterdir())

    def overwrite():
        c = C(_tmp() / "ckpt")
        c.save(3, {"v": 1})
        c.save(3, {"v": 2})
        return (c.steps(), c.load(3))

    _scenarios("Checkpointer", [
        ("keeps only the last N checkpoints", rotation,
         ([3, 4, 5], ["ckpt_000003.json", "ckpt_000004.json", "ckpt_000005.json"]),
         "file names are ckpt_{step:06d}.json. Delete the oldest ones after saving"),
        ("latest() returns (step, state)", latest, (20, {"w": 2.0}), None),
        ("latest() with no checkpoints", lambda: C(_tmp() / "empty").latest(), None, None),
        ("a new Checkpointer finds existing files", persists, (7, {"lr": 0.01}), "read the folder in steps(), don't only remember what this object saved"),
        ("other files in the folder are left alone", ignores_other_files, ["ckpt_000002.json", "notes.txt"], None),
        ("saving the same step again overwrites it", overwrite, ([3], {"v": 2}), None),
        ("keep_last=0 is invalid", lambda: C(_tmp(), keep_last=0), Raises(ValueError), None),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `parse_value`

In [2]:
MISSING_MARKERS = {"", "na", "n/a", "nan", "null", "none", "?", "-"}


def parse_value(text: str):
    text = text.strip()
    lower = text.lower()
    if lower in MISSING_MARKERS:
        return None
    if lower in ("true", "false"):
        return lower == "true"
    try:
        return int(text)
    except ValueError:
        pass
    try:
        return float(text)
    except ValueError:
        return text

In [3]:
check_parse_value(parse_value)

PASS  all 12 tests passed for parse_value


The order of the checks is the whole exercise:

1. **Missing markers first.** `float("nan")` succeeds and returns a NaN, and `float("NaN")` too. If you tried converting to a number first, the text `"nan"` would become a float NaN instead of `None`.
2. **Booleans before numbers**, since neither `int` nor `float` accepts them anyway, but checking them early keeps the logic obvious.
3. **`int` before `float`.** `float("42")` succeeds too, but returns `42.0`. Trying `int` first keeps whole numbers as integers, which matters for things like IDs, counts and years.

"Try it and catch the error" (`try: int(text) except ValueError`) is the normal Python way to check whether a string is a number. Writing your own rules for what counts as a number is surprisingly hard: `"1e-3"`, `"-7"`, `" 3.0"` and `"inf"` are all valid floats.

pandas' `read_csv` does this same kind of guessing for every column, and it's a common source of surprises. A column of ZIP codes like `"01234"` becomes the integer `1234`, losing the leading zero. When types matter, tell the reader explicitly what each column is, which is what the schema in exercise 10 does.

### Exercise 2: `count_lines`

In [4]:
def count_lines(path) -> int:
    with open(path, encoding="utf-8") as f:
        return sum(1 for _ in f)

In [5]:
check_count_lines(count_lines)

PASS  all 5 tests passed for count_lines


Looping over a file object yields one line at a time, including a last line that has no newline at the end, so counting the iterations gives exactly the right answer. `sum(1 for _ in f)` counts without building a list, so memory use stays flat no matter how big the file is.

Counting `"\n"` characters instead (`text.count("\n")`) gets the last line wrong when the file doesn't end with a newline, and `f.read()` loads the whole file into memory first.

`open()` accepts both strings and `Path` objects, so there's no need to convert.

This is a handy first check on a new dataset: if a CSV has 1,000,001 lines, you expect 1,000,000 rows plus a header. If your loader returns fewer rows than that, something was skipped, and you want to know why.

### Exercise 3: `save_json`

In [6]:
import json
from pathlib import Path


def save_json(obj, path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    text = json.dumps(obj, indent=2, ensure_ascii=False)
    path.write_text(text + "\n", encoding="utf-8")
    return path

In [7]:
check_save_json(save_json)

PASS  all 5 tests passed for save_json


`Path(path)` at the top lets the function accept either a string or a `Path`, and from then on we can use `Path` methods. It's a common first line in functions that take paths.

`ensure_ascii=False` keeps non-English characters readable in the file. The default (`True`) escapes them as `\u00e9`, which is valid JSON but hard to read. Since we're writing with `encoding="utf-8"`, the characters are stored correctly either way.

The trailing newline is a small convention that matters more than it looks. Many tools expect text files to end with one, and without it, `git diff` shows a "No newline at end of file" warning every time the file changes.

A helper like this, used everywhere in a project, means every JSON file is formatted the same way, which keeps `git diff` output clean when results change between runs.

### Exercise 4: `find_files`

In [8]:
from pathlib import Path


def find_files(folder, suffix: str) -> list[str]:
    folder = Path(folder)
    suffix = suffix.lower()
    return sorted(
        path.relative_to(folder).as_posix()
        for path in folder.rglob("*")
        if path.is_file() and path.suffix.lower() == suffix
    )

In [9]:
check_find_files(find_files)

PASS  all 5 tests passed for find_files


`rglob("*")` walks through every file and folder below `folder`, at any depth. The filter keeps only files whose suffix matches, compared in lowercase.

`relative_to(folder)` turns `/content/dataset/sub/c.csv` into `sub/c.csv`, and `as_posix()` writes it with forward slashes even on Windows. That matters if you save these paths into a file (like a list of training images), because the file then works on every operating system.

Sorting isn't just tidiness. `rglob` returns files in whatever order the file system gives them, which can differ between machines. If you build a dataset from that list and then shuffle it with a fixed seed, an unsorted list means a different "random" split on each machine, and your results stop being reproducible. Sorting first fixes the order.

## Medium

### Exercise 5: `read_csv_records`

In [10]:
import csv


def read_csv_records(path) -> list[dict]:
    with open(path, encoding="utf-8-sig", newline="") as f:
        return [
            {key: parse_value(value) for key, value in row.items()}
            for row in csv.DictReader(f)
        ]

In [11]:
check_read_csv_records(read_csv_records)

PASS  all 3 tests passed for read_csv_records


`encoding="utf-8-sig"` strips a BOM if there is one and does nothing otherwise, so it's safe for every UTF-8 file. Without it, the first column of an Excel-made file is called `"\ufeffid"`, and `row["id"]` raises a `KeyError` on a column you can plainly see in the file.

`newline=""` is what the `csv` documentation asks for, so the `csv` module can handle line breaks inside quoted values itself.

The dict comprehension inside the list comprehension converts each row as it's read. The `with` block closes the file when the function returns, after the list has been fully built, so there's no risk of reading from a closed file.

For very large files you'd turn this into a generator (with `yield` instead of building a list), so rows are processed one at a time.

### Exercise 6: `write_csv`

In [12]:
import csv


def write_csv(records: list[dict], path) -> int:
    columns = list(dict.fromkeys(key for record in records for key in record))
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        for record in records:
            writer.writerow({k: ("" if v is None else v) for k, v in record.items()})
    return len(records)

In [13]:
check_write_csv(write_csv)

PASS  all 3 tests passed for write_csv


`dict.fromkeys(...)` collects the unique keys in first-seen order, the trick from Module 2.

`DictWriter` already fills missing keys with an empty string (that's its `restval` setting), so only `None` needs converting. Without the conversion, `None` would be written as the text `None`, and the next person to read the file would get the string `"None"` instead of a missing value.

The check is `v is None`, not `if not v`. A truthiness check would also turn `0` and `False` into empty fields, silently deleting real data.

Writing with `DictWriter` also protects against the reverse of the field-count problem from the lesson: it quotes any value containing a comma, quote or newline, so the file can always be read back correctly.

### Exercise 7: `missing_report`

In [14]:
def missing_report(records: list[dict]) -> list[tuple]:
    if not records:
        return []
    columns = dict.fromkeys(key for record in records for key in record)
    n = len(records)
    report = []
    for col in columns:
        n_missing = sum(record.get(col) is None for record in records)
        report.append((col, n_missing, n_missing / n))
    return sorted(report, key=lambda r: (-r[1], r[0]))

In [15]:
check_missing_report(missing_report)

PASS  all 3 tests passed for missing_report


`record.get(col) is None` handles both kinds of missing in one check: `.get` returns `None` for an absent key, and a stored `None` is also `None`.

The sort key `(-count, name)` is the tuple-key trick from Module 2: highest count first, alphabetical for ties.

The empty-records check comes first so we never divide by zero.

This is usually the second thing to run on a new dataset, right after counting rows. It tells you at a glance which columns are usable as they are, which need filling in, and which are so empty they're better dropped. As a rough guide, a column that's 90% missing rarely helps a model, while one that's 2% missing is usually worth filling.

### Exercise 8: `fill_missing`

In [16]:
from collections import Counter


def fill_missing(train: list[dict], test: list[dict], column: str, strategy: str = "median"):
    values = [r[column] for r in train if r.get(column) is not None]
    if not values:
        raise ValueError(f"no values in training data for column {column!r}")

    if strategy == "mean":
        fill = sum(values) / len(values)
    elif strategy == "median":
        s = sorted(values)
        mid = len(s) // 2
        fill = s[mid] if len(s) % 2 else (s[mid - 1] + s[mid]) / 2
    elif strategy == "mode":
        fill = Counter(values).most_common(1)[0][0]
    else:
        raise ValueError(f"unknown strategy {strategy!r}; use 'mean', 'median' or 'mode'")

    def filled(records):
        return [{**r, column: fill if r.get(column) is None else r[column]} for r in records]

    return filled(train), filled(test), fill

In [17]:
check_fill_missing(fill_missing)

PASS  all 7 tests passed for fill_missing


The function's signature makes the leakage rule hard to break: it takes train and test separately and only ever looks at `train` to compute the fill value. If it took one combined list, it would be easy to compute the median over everything by accident.

`{**r, column: ...}` builds a new dict with every key from `r`, then overrides one. The original records are never touched, the same merging trick as the configs in Module 2.

For `mode`, `Counter.most_common` keeps ties in first-seen order, so ties go to whichever value appeared first, as the exercise asked.

Which strategy to use? The **median** is the usual default for numbers, because a few extreme values don't pull it around the way they pull the mean. The **mode** is the only one of the three that works for categories. For anything important, it's worth also adding a column like `mass_was_missing` (true or false), because the fact that a value was missing can itself be informative to a model.

### Exercise 9: `read_jsonl`

In [18]:
import json


def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            if not line.strip():
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(f"line {line_number}: {e}") from e

In [19]:
check_read_jsonl(read_jsonl)

PASS  all 3 tests passed for read_jsonl


`json.JSONDecodeError` is a subclass of `ValueError`, so the raw error would technically satisfy "raise a ValueError". But its message describes a position *within that one line* ("line 1 column 9"), which is useless in a file with a million lines. Wrapping it with the real line number turns it into a message you can act on: open the file, go to line 3, see the problem.

`raise ... from e` keeps the original error attached, so the full traceback still shows the underlying JSON error as well.

One subtlety: with a generator, the `with` block stays open between `yield`s. The file is closed when the generator finishes, or when it's garbage collected if the caller stops early. For the usual "loop over all records" case, that's exactly what you want.

Should a bad line stop everything, or be skipped? For a training log you wrote yourself, a bad line means something is wrong and stopping is right. For a huge scraped dataset where a few broken lines are expected, skipping and counting them (as in Module 4's `parse_scores`) is more practical. Make it a deliberate choice either way.

## Hard

### Exercise 10: `validate_records`

In [20]:
TYPES = {"str": (str,), "int": (int,), "float": (int, float)}


def check_value(value, rule) -> str | None:
    """Return an error code for one value, or None if it's fine."""
    if value is None:
        return "missing" if rule.get("required", True) else None
    if isinstance(value, bool) or not isinstance(value, TYPES[rule["type"]]):
        return "type"
    if "allowed" in rule and value not in rule["allowed"]:
        return "not_allowed"
    if "min" in rule and value < rule["min"]:
        return "below_min"
    if "max" in rule and value > rule["max"]:
        return "above_max"
    return None


def validate_records(records: list[dict], schema: dict):
    valid, errors = [], []
    for i, record in enumerate(records):
        record_errors = []
        for column, rule in schema.items():
            code = check_value(record.get(column), rule)
            if code:
                record_errors.append((i, column, code))
        if record_errors:
            errors.extend(record_errors)
        else:
            valid.append(record)
    return valid, errors

In [21]:
check_validate_records(validate_records)

PASS  all 4 tests passed for validate_records


Splitting out `check_value` keeps each function simple: one decides what's wrong with a single value, the other loops and collects. Each check returns as soon as it finds a problem, which naturally gives the "skip the remaining checks" behaviour.

The `bool` check deserves a closer look. In Python, `bool` is a subclass of `int`, so `isinstance(True, int)` is `True`, and `True + True` is `2`. Without the explicit `isinstance(value, bool)` check, a body mass of `True` would pass as the number 1 and then fail the `min` check with a misleading error. Here it correctly reports a type error. Since a `bool` is never a valid `str`, `int` or `float` in this schema, one check covers every column type.

Reporting **every** error, rather than stopping at the first, is what makes a validator useful on a real dataset. You want to see "312 rows have a body mass below the minimum" in one run, not fix one row, rerun, fix the next, and so on.

Libraries like `pydantic`, `pandera` and Great Expectations do this at scale, with much richer rules. The idea is the same: write down what valid data looks like, check every row, report everything that doesn't match.

### Exercise 11: `deduplicate`

In [22]:
def _normalize(value):
    return value.strip().lower() if isinstance(value, str) else value


def deduplicate(records: list[dict], keys: list[str] | None = None):
    if keys is None:
        keys = list(dict.fromkeys(k for r in records for k in r))
    seen = set()
    unique = []
    for record in records:
        signature = tuple(_normalize(record.get(k)) for k in keys)
        if signature not in seen:
            seen.add(signature)
            unique.append(record)
    return unique, len(records) - len(unique)

In [23]:
check_deduplicate(deduplicate)

PASS  all 5 tests passed for deduplicate


Each record is reduced to a **signature**: a tuple of its normalized key values. Tuples are hashable, so signatures can go in a set, and checking whether we've seen one before is O(1). Comparing every record against every other record instead would be O(n²), which is far too slow for a dataset of a million rows.

The normalization only affects the signature. The record that goes into `unique` is the original one, so we don't silently change anyone's data while deduplicating it.

Which columns to compare on is a real decision. Deduplicating on an ID column finds the same record entered twice. Deduplicating on all the *feature* columns finds something different and just as important: identical examples, which, if they land in both the training and test sets, inflate your test score. Image and text datasets scraped from the web are full of these, and large ML teams spend real effort removing them.

### Exercise 12: `Checkpointer`

In [24]:
import json
import re
from pathlib import Path


class Checkpointer:
    PATTERN = re.compile(r"ckpt_(\d+)\.json")

    def __init__(self, folder, keep_last: int = 3):
        if keep_last < 1:
            raise ValueError(f"keep_last must be at least 1, got {keep_last}")
        self.folder = Path(folder)
        self.folder.mkdir(parents=True, exist_ok=True)
        self.keep_last = keep_last

    def _path(self, step: int) -> Path:
        return self.folder / f"ckpt_{step:06d}.json"

    def steps(self) -> list[int]:
        found = []
        for path in self.folder.iterdir():
            match = self.PATTERN.fullmatch(path.name)
            if match and path.is_file():
                found.append(int(match.group(1)))
        return sorted(found)

    def save(self, step: int, state: dict) -> Path:
        path = self._path(step)
        tmp = path.with_name(path.name + ".tmp")
        tmp.write_text(json.dumps({"step": step, "state": state}), encoding="utf-8")
        tmp.replace(path)
        for old in self.steps()[:-self.keep_last]:
            self._path(old).unlink()
        return path

    def load(self, step: int) -> dict:
        return json.loads(self._path(step).read_text(encoding="utf-8"))["state"]

    def latest(self):
        steps = self.steps()
        if not steps:
            return None
        return steps[-1], self.load(steps[-1])

In [25]:
check_Checkpointer(Checkpointer)

PASS  all 7 tests passed for Checkpointer


**The folder is the source of truth.** `steps()` reads the folder every time instead of keeping a list in memory. That's what lets a brand-new `Checkpointer`, in a new process after a crash, find the checkpoints from the previous run, which is the whole point of checkpoints.

**Recognising our own files.** A regular expression (`re`) matches exactly `ckpt_` + digits + `.json`, so `notes.txt`, or a half-written `ckpt_000005.json.tmp`, is never mistaken for a checkpoint, and never deleted. `fullmatch` requires the whole name to match, not just part of it.

**Writing safely.** The state is written to a temporary file first, then `replace` renames it over the real name. Renaming a file within the same folder is atomic on all common operating systems: it either fully happens or doesn't happen at all. So if the program crashes halfway through writing, you're left with a stray `.tmp` file and your previous checkpoint intact, rather than a corrupted checkpoint. That matters a lot when the thing being saved is a week of training.

**Rotation.** `self.steps()[:-self.keep_last]` is every step except the newest `keep_last`, and those files are deleted with `unlink`. Since `keep_last` is at least 1, the slice is never `[:-0]`, which would be the empty-slice trap from Module 1 in reverse: `steps[:-0]` is `steps[:0]`, an empty list, so nothing would ever be deleted.

Real frameworks store checkpoints in binary formats (PyTorch's `.pt` or `safetensors`), because model weights are huge. The logic of naming, rotating, atomically writing and resuming is exactly this.

## What's next

The mini project in `04_mini_project.ipynb` goes from the raw penguin CSV to a trained species classifier, saves the model as JSON, and loads it back to make predictions.